In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
import joblib




#Load cleaned dataset

df = pd.read_csv('/Users/olgabencomo/Desktop/Proyectos Portafolio/Fraud Detection/data/processed/cleaned_dataset.csv')

In [10]:
df.head()

,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud,hour_sin,hour_cos
0,9608,363,4922.587542,ATM,Travel,TR,12,0.992347,0.947908,1,1.224647e-16,-1.000000
1,456,692,48.018303,QR,Food,US,21,0.168571,0.224057,0,-7.071068e-01,0.707107
2,4747,587,136.881960,Online,Travel,TR,14,0.296127,0.125058,0,-5.000000e-01,-0.866025
3,6934,445,80.534719,POS,Clothing,TR,23,0.124801,0.159243,0,-2.588190e-01,0.965926
4,1646,729,120.041158,Online,Grocery,FR,16,0.098129,0.027542,0,-8.660254e-01,-0.500000


In [11]:
X = df.drop(columns=['is_fraud'])
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [12]:
#Apply One Hot Encoding to categorical features

categorical_cols = ['transaction_type', 'merchant_category', 'country']

encoder = OneHotEncoder(drop='first', sparse_output=False)

encoder.fit(X_train[categorical_cols])
joblib.dump(encoder, '/Users/olgabencomo/Desktop/Proyectos Portafolio/Fraud Detection/models/encoder.pkl')

train_ohe = encoder.transform(X_train[categorical_cols])
test_ohe  = encoder.transform(X_test[categorical_cols])

train_ohe = pd.DataFrame(train_ohe,
                         columns=encoder.get_feature_names_out(),
                         index=X_train.index)

test_ohe = pd.DataFrame(test_ohe,
                        columns=encoder.get_feature_names_out(),
                        index=X_test.index)

X_train = pd.concat([X_train.drop(columns=categorical_cols), train_ohe], axis=1)
X_test  = pd.concat([X_test.drop(columns=categorical_cols),  test_ohe], axis=1)


In [13]:
X_train

,transaction_id,user_id,amount,hour,device_risk_score,ip_risk_score,hour_sin,hour_cos,transaction_type_Online,transaction_type_POS,transaction_type_QR,merchant_category_Electronics,merchant_category_Food,merchant_category_Grocery,merchant_category_Travel,country_FR,country_NG,country_TR,country_UK,country_US
1771,2660,911,97.887042,22,0.073841,0.153680,-5.000000e-01,8.660254e-01,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5149,823,442,94.161800,11,0.250203,0.103801,2.588190e-01,-9.659258e-01,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
456,2102,707,80.783739,8,0.076352,0.261951,8.660254e-01,-5.000000e-01,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1343,6699,98,51.411597,15,0.127681,0.028990,-7.071068e-01,-7.071068e-01,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5957,6041,254,121.744317,9,0.205196,0.234584,7.071068e-01,-7.071068e-01,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4102,3920,152,101.406776,22,0.240656,0.187714,-5.000000e-01,8.660254e-01,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3694,7739,905,152.833097,6,0.238990,0.056883,1.000000e+00,6.123234e-17,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
6986,4814,555,119.218386,12,0.297990,0.060788,1.224647e-16,-1.000000e+00,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
6651,4113,272,115.386763,19,0.267263,0.101328,-9.659258e-01,2.588190e-01,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [14]:
df_train = X_train.copy()
df_train['is_fraud'] = y_train


In [15]:
corr_target = df_train.corr(numeric_only=True)['is_fraud'].sort_values(ascending=False)
corr_target

is_fraud                         1.000000
ip_risk_score                    0.871277
device_risk_score                0.871099
amount                           0.627093
country_NG                       0.435312
transaction_id                   0.377674
hour_sin                         0.115910
hour_cos                         0.109668
merchant_category_Travel         0.002940
transaction_type_Online          0.002753
user_id                          0.000957
transaction_type_QR             -0.001126
merchant_category_Grocery       -0.004841
merchant_category_Electronics   -0.005007
transaction_type_POS            -0.008512
country_UK                      -0.009010
merchant_category_Food          -0.012377
country_US                      -0.012786
country_TR                      -0.021176
country_FR                      -0.036184
hour                            -0.183845
Name: is_fraud, dtype: float64

In [16]:
#Drop Unused columns

cols_to_drop = ['transaction_id', 'user_id', 'hour','ip_risk_score', 'device_risk_score']

X_train.drop(columns=cols_to_drop, inplace=True)
X_test.drop(columns=cols_to_drop, inplace=True)

## Feature Selection & Column Removal

After applying encoding, a correlation analysis was performed between all numeric variables 
and the target variable `is_fraud`, in order to identify columns that could represent 
a **data leakage** risk prior to model training.

The following columns were removed:
- `transaction_id` and `user_id`: unique identifiers with no predictive value.
- `hour`: replaced by its cyclical representations `hour_sin` and `hour_cos`.
- `ip_risk_score` and `device_risk_score`: showed an unusually high correlation with `is_fraud`.

This last point deserves special attention. A very high correlation between a feature and 
the target is not always a good sign — in many cases, it is a red flag. In a fraud detection 
project, variables such as `ip_risk_score` or `device_risk_score` may have been constructed 
using the knowledge of whether a transaction was fraudulent or not. This means the model 
would be learning to predict fraud using information that would not be available at the time 
of the transaction in a real production environment, which leads to artificially inflated 
metrics (99% accuracy, AUC close to 1.0) that fail completely once deployed.

For this reason, keeping a variable solely because it has a high correlation with the target 
would be a mistake. The right question is not how correlated is this variable with fraud?
but rather would this information be available at the exact moment the decision needs to be made? 
If the answer is no, or if there is any uncertainty about how the variable was originally built, 
the safest choice is to exclude it from the model.

In [17]:
scaler = StandardScaler()
scaler.fit(X_train[['amount']])
joblib.dump(scaler, '/Users/olgabencomo/Desktop/Proyectos Portafolio/Fraud Detection/models/scaler.pkl')

X_train['amount_scaled'] = scaler.transform(X_train[['amount']])
X_test['amount_scaled']  = scaler.transform(X_test[['amount']])

X_train.drop(columns=['amount'], inplace=True)
X_test.drop(columns=['amount'], inplace=True)


In [18]:
#Export X_train and X_test

X_train.to_csv('/Users/olgabencomo/Desktop/Proyectos Portafolio/Fraud Detection/data/train dataset/X_train.csv', index=False)
X_test.to_csv('/Users/olgabencomo/Desktop/Proyectos Portafolio/Fraud Detection/data/test dataset/X_test.csv', index=False)

In [19]:
#Export y_train and y_test

y_train.to_csv('/Users/olgabencomo/Desktop/Proyectos Portafolio/Fraud Detection/data/train dataset/y_train.csv', index=False)
y_test.to_csv('/Users/olgabencomo/Desktop/Proyectos Portafolio/Fraud Detection/data/test dataset/y_test.csv', index=False)